# Branch 3 — Combined Models — v8.1 Paper-Ready + Holm Correction T4 x2


### v8 paper-ready corrections

- identical matched observations and chronological splits;
- validation-only threshold/model selection;
- controlled fixed-settings numerical→combined comparisons;
- separate validation-selected best-system comparison;
- company-cluster paired bootstrap for balanced-accuracy gaps;
- exact McNemar as a supplementary overall-accuracy test;
- row-level correctness exports and one unified paper results workbook.


In [ ]:
# SIC 3674 multimodal dataset input — Kaggle + Colab aware
from pathlib import Path

TARGET_DATASET_NAMES = [
    "sic3674_multimodal_model_rows.parquet",
    "sic3674_multimodal_model_rows.csv",
]

def resolve_sic3674_dataset_path():
    # 1) Kaggle attached datasets / working files
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            for filename in TARGET_DATASET_NAMES:
                matches = list(root.rglob(filename))
                if matches:
                    return matches[0]

    # 2) Colab / Google Drive
    candidates = [
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.csv"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate sic3674_multimodal_model_rows.parquet/csv. "
        "Attach the dataset to Kaggle or place it in the Colab/Drive sic3674_output folder."
    )

SIC3674_DATA_PATH = resolve_sic3674_dataset_path()
print("Using SIC 3674 dataset:", SIC3674_DATA_PATH)


In [ ]:
!pip -q install pyarrow openpyxl xgboost sentence-transformers transformers accelerate beautifulsoup4 requests tqdm scipy

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH: Path | None = None
NEUTRAL_BAND = 0.02
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)


OUTPUT_DIR = Path(
    '/kaggle/working/datasets/pjbob1/semiconductor_branch_combined_v8_1_holm_paper_ready_t4x2'
    if Path('/kaggle/working').exists()
    else '/content/semiconductor_branch_combined_v8_1_holm_paper_ready_t4x2'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DATA_PATH: Path | None = None
SEC_USER_AGENT = 'YOUR NAME your.email@example.com'
DOWNLOAD_SEC_TEXT_FROM_EDGAR = True
TEXT_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MIN_TEXT_CHARS = 500
MAX_DOCUMENT_CHARS = 120_000
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MAX_CHUNKS_PER_FILING = 16
TEXT_BATCH_SIZE = 32
TEXT_PCA_COMPONENTS = 32
TEXT_PCA_GRID = [16, 32, 64, 128]
RUN_LOCO = False
RUN_FINBERT_SENTIMENT = True
FINBERT_MODEL_NAME = 'ProsusAI/finbert'
TEXT_CACHE_DIR = OUTPUT_DIR/'sec_text_cache'
TEXT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# GPU diagnostics. On Kaggle T4 x2 this should report two CUDA devices.
try:
    import torch
    GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
    GPU_DEVICES = [f"cuda:{i}" for i in range(GPU_COUNT)]
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", GPU_COUNT)
    for i in range(GPU_COUNT):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    if GPU_COUNT >= 2:
        print("T4 x2 / multi-GPU mode enabled for text inference.")
    elif GPU_COUNT == 1:
        print("Single-GPU mode enabled.")
    else:
        print("No CUDA GPU detected; text inference will use CPU.")
except Exception as exc:
    GPU_COUNT = 0
    GPU_DEVICES = []
    print("GPU detection failed:", repr(exc))

## Load the same dataset used by v6.1

In [ ]:
def discover_data_path() -> Path:
    # Prefer the resolved SIC 3674 multimodal dataset.
    if "SIC3674_DATA_PATH" in globals() and Path(SIC3674_DATA_PATH).exists():
        return Path(SIC3674_DATA_PATH)

    candidates = [
        Path('/content/drive/MyDrive/sec_research_semiconductor/semiconductor_sec_numeric_text.parquet'),
        Path('/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet'),
        Path('/content/semiconductor_sec_numeric_text.parquet'),
        Path('/content/sec_experiment_semiconductor.parquet'),
        Path('semiconductor_sec_numeric_text.parquet'),
        Path('semiconductor_sec_numeric_text.csv'),
        Path('sec_experiment_semiconductor.parquet'),
        Path('model_dataset.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            print('Found dataset automatically:', candidate)
            return candidate

    try:
        from google.colab import files
        print('Upload the same parquet/CSV dataset used by the experiment.')
        uploaded = files.upload()
        choices = [
            Path(name) for name in uploaded
            if Path(name).suffix.lower() in {'.parquet', '.csv'}
        ]
        if not choices:
            raise FileNotFoundError('Upload a .parquet or .csv file.')
        return choices[0]
    except ImportError as exc:
        raise FileNotFoundError(
            'Dataset not found. Attach it to Kaggle or set DATA_PATH explicitly.'
        ) from exc

resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()
if resolved_path.suffix.lower() == '.parquet':
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == '.csv':
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

print('Loaded:', resolved_path)
print('Rows:', len(raw), '| Columns:', len(raw.columns))


## Rebuild target exactly as v6.1

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

## Financial feature engineering

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

## Chronological split

In [ ]:
# Use an existing split only when it contains all three required groups.
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "The existing split column does not contain all three groups. "
            "Rebuilding the split chronologically."
        )

    data["split"] = pd.NA

    # Use the most conservative available timestamp:
    # when the target became knowable, then the feature cutoff, then quarter end.
    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required to create "
            "train, validation, and test splits."
        )

    # Automatic chronological 60% / 20% / 20% split by distinct dates.
    train_position = max(0, min(len(eligible_dates) - 3, int(len(eligible_dates) * 0.60) - 1))
    validation_position = max(
        train_position + 1,
        min(len(eligible_dates) - 2, int(len(eligible_dates) * 0.80) - 1),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)

model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()
model_data["target_clean"] = model_data["target_clean"].astype(int)

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Could not create these splits: {sorted(missing_splits)}. "
        "The dataset may have too few labeled dates after applying the "
        "neutral band."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[model_data["split"] == split_name]
    if split_frame["target_clean"].nunique() < 2:
        print(
            f"Warning: {split_name} contains only one target class after "
            f"applying the {NEUTRAL_BAND:.1%} neutral band. "
            "Try reducing NEUTRAL_BAND to 0.01 if model fitting fails."
        )

## Numerical feature list

In [ ]:
candidate_numeric_features = [
    'revenue_yoy_growth', 'revenue_momentum', 'sequential_revenue_growth',
    'gross_margin', 'gross_margin_change', 'operating_margin',
    'operating_margin_change', 'cash_flow_margin', 'cash_flow_margin_change',
    'inventory_to_ttm_revenue', 'inventory_to_ttm_revenue_change',
    'inventory_yoy_growth', 'inventory_revenue_growth_gap',
    'receivables_to_ttm_revenue', 'receivables_to_ttm_revenue_change',
    'receivables_yoy_growth', 'receivables_revenue_growth_gap',
    'capex_to_revenue', 'capex_to_revenue_change', 'capex_yoy_growth',
    'log_assets', 'liabilities_to_assets',
]

candidate_numeric_features += [
    f'{feature}_relative_to_sector' for feature in relative_base_features
]

numeric_features = [
    c for c in dict.fromkeys(candidate_numeric_features)
    if c in model_data.columns
    and model_data[c].notna().sum() >= 10
    and model_data[c].nunique(dropna=True) > 1
]

categorical_features = [
    c for c in ['fiscal_quarter', 'semiconductor_subgroup']
    if c in model_data.columns and model_data[c].notna().any()
]
feature_columns = numeric_features + categorical_features
print('Numerical features:', len(numeric_features))
print('Categorical features:', categorical_features)
display(pd.DataFrame({'feature': feature_columns}))

## Attach SEC filing text

In [ ]:
import hashlib
import html as html_lib
import json
import re
import time
from collections.abc import Iterable

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


TEXT_COLUMN_CANDIDATES = [
    "sec_text",
    "filing_text",
    "document_text",
    "mda_text",
    "management_discussion_text",
    "risk_factors_text",
    "risk_text",
]


def normalize_accession(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def combine_available_text_columns(frame: pd.DataFrame) -> pd.Series:
    available = [
        column for column in TEXT_COLUMN_CANDIDATES
        if column in frame.columns
    ]
    if not available:
        return pd.Series("", index=frame.index, dtype="object")

    combined = (
        frame[available]
        .fillna("")
        .astype(str)
        .apply(
            lambda row: "\n\n".join(
                value.strip()
                for value in row
                if value and value.strip()
            ),
            axis=1,
        )
    )
    return combined


def discover_text_data_path() -> Path | None:
    candidates = [
        Path("/content/drive/MyDrive/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/drive/MyDrive/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("/content/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("sec_filing_text.parquet"),
        Path("sec_filing_text.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError("Text dataset must be a .parquet or .csv file.")


def merge_external_text(
    base: pd.DataFrame,
    text_frame: pd.DataFrame,
) -> pd.DataFrame:
    external = text_frame.copy()

    for frame in [base, external]:
        if "quarter_end" in frame.columns:
            frame["quarter_end"] = pd.to_datetime(
                frame["quarter_end"], errors="coerce"
            )
        if "filing_date" in frame.columns:
            frame["filing_date"] = pd.to_datetime(
                frame["filing_date"], errors="coerce"
            )
        if "accession_number" in frame.columns:
            frame["accession_number"] = frame[
                "accession_number"
            ].map(normalize_accession)
        if "cik" in frame.columns:
            frame["cik"] = frame["cik"].astype(str)
        if "ticker" in frame.columns:
            frame["ticker"] = frame["ticker"].astype(str)

    external["external_sec_text"] = combine_available_text_columns(
        external
    )

    if "accession_number" in base.columns and "accession_number" in external.columns:
        merge_keys = ["accession_number"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["cik", "quarter_end"]
    ):
        merge_keys = ["cik", "quarter_end"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["ticker", "quarter_end"]
    ):
        merge_keys = ["ticker", "quarter_end"]
    else:
        raise ValueError(
            "The text dataset must share accession_number, "
            "(cik, quarter_end), or (ticker, quarter_end) with the "
            "experiment dataset."
        )

    keep_columns = merge_keys + ["external_sec_text"]
    if "filing_date" in external.columns:
        keep_columns.append("filing_date")

    external = (
        external[keep_columns]
        .drop_duplicates(subset=merge_keys, keep="last")
    )

    merged = base.merge(
        external,
        on=merge_keys,
        how="left",
        suffixes=("", "_text_source"),
    )

    merged["sec_text"] = np.where(
        merged["sec_text"].fillna("").str.len()
        >= merged["external_sec_text"].fillna("").str.len(),
        merged["sec_text"].fillna(""),
        merged["external_sec_text"].fillna(""),
    )
    return merged


def clean_filing_html(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    # The project is using the narrative writing; large XBRL tables add
    # many repeated numbers and labels without much prose.
    for table in soup.find_all("table"):
        table.decompose()

    text = soup.get_text(" ")
    text = html_lib.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_longest_section(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
    minimum_chars: int = 500,
    maximum_chars: int = 100_000,
) -> str:
    starts = []
    for pattern in start_patterns:
        starts.extend(re.finditer(pattern, text, flags=re.I))

    ends = []
    for pattern in end_patterns:
        ends.extend(re.finditer(pattern, text, flags=re.I))

    candidates: list[str] = []
    for start in starts:
        possible_ends = [
            end for end in ends
            if end.start() > start.end() + minimum_chars
        ]
        if not possible_ends:
            continue
        end = min(possible_ends, key=lambda match: match.start())
        candidate = text[start.start():end.start()].strip()
        if minimum_chars <= len(candidate) <= maximum_chars:
            candidates.append(candidate)

    return max(candidates, key=len) if candidates else ""


def extract_narrative_sections(clean_text: str) -> str:
    mda = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+7[\.\:\-\s]+management[’']?s?\s+discussion",
            r"\bitem\s+2[\.\:\-\s]+management[’']?s?\s+discussion",
        ],
        end_patterns=[
            r"\bitem\s+7a[\.\:\-\s]+",
            r"\bitem\s+8[\.\:\-\s]+financial",
            r"\bitem\s+3[\.\:\-\s]+quantitative",
            r"\bitem\s+4[\.\:\-\s]+controls",
        ],
    )

    risks = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
        ],
        end_patterns=[
            r"\bitem\s+1b[\.\:\-\s]+",
            r"\bitem\s+1c[\.\:\-\s]+",
            r"\bitem\s+2[\.\:\-\s]+",
        ],
    )

    sections = []
    if mda:
        sections.append("MANAGEMENT DISCUSSION AND ANALYSIS\n" + mda)
    if risks:
        sections.append("RISK FACTORS\n" + risks)

    if sections:
        return "\n\n".join(sections)[:MAX_DOCUMENT_CHARS]

    # Fallback when filing headings differ from the standard patterns.
    return clean_text[:MAX_DOCUMENT_CHARS]


def valid_sec_user_agent(user_agent: str) -> bool:
    lowered = user_agent.lower()
    return (
        "your name" not in lowered
        and "example.com" not in lowered
        and "@" in user_agent
        and len(user_agent.strip()) >= 8
    )


class EdgarTextDownloader:
    def __init__(self, user_agent: str, cache_dir: Path):
        if not valid_sec_user_agent(user_agent):
            raise ValueError(
                "Replace SEC_USER_AGENT with your real name and email "
                "before downloading from EDGAR."
            )

        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": user_agent,
                "Accept-Encoding": "gzip, deflate",
            }
        )
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.submission_cache: dict[str, dict[str, str]] = {}

    def get_json(self, url: str) -> dict:
        response = self.session.get(url, timeout=60)
        response.raise_for_status()
        time.sleep(0.20)
        return response.json()

    def get_text(self, url: str) -> str:
        response = self.session.get(url, timeout=90)
        response.raise_for_status()
        time.sleep(0.20)
        return response.text

    def _add_submission_rows(
        self,
        mapping: dict[str, str],
        payload: dict,
    ) -> None:
        accessions = payload.get("accessionNumber", [])
        primary_documents = payload.get("primaryDocument", [])
        for accession, document in zip(
            accessions, primary_documents, strict=False
        ):
            if accession and document:
                mapping[str(accession)] = str(document)

    def submission_map(self, cik: str) -> dict[str, str]:
        cik_key = str(int(float(cik))).zfill(10)
        if cik_key in self.submission_cache:
            return self.submission_cache[cik_key]

        payload = self.get_json(
            f"https://data.sec.gov/submissions/CIK{cik_key}.json"
        )
        mapping: dict[str, str] = {}
        self._add_submission_rows(
            mapping,
            payload.get("filings", {}).get("recent", {}),
        )

        # Older filings can be stored in additional submission JSON files.
        for file_info in payload.get("filings", {}).get("files", []):
            file_name = file_info.get("name")
            if not file_name:
                continue
            old_payload = self.get_json(
                f"https://data.sec.gov/submissions/{file_name}"
            )
            self._add_submission_rows(mapping, old_payload)

        self.submission_cache[cik_key] = mapping
        return mapping

    def primary_document(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        if supplied_document is not None and not pd.isna(supplied_document):
            supplied = str(supplied_document).strip()
            if supplied:
                return supplied

        mapping = self.submission_map(cik)
        document = mapping.get(accession)
        if document:
            return document

        # Final fallback: choose the largest non-index HTML document.
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        index_payload = self.get_json(
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/index.json"
        )
        items = (
            index_payload.get("directory", {}).get("item", [])
        )
        html_items = [
            item for item in items
            if str(item.get("name", "")).lower().endswith(
                (".htm", ".html")
            )
            and "-index." not in str(item.get("name", "")).lower()
            and "filingsummary" not in str(item.get("name", "")).lower()
        ]
        if not html_items:
            raise FileNotFoundError(
                f"No primary HTML document found for {accession}."
            )
        chosen = max(
            html_items,
            key=lambda item: int(item.get("size", 0) or 0),
        )
        return str(chosen["name"])

    def filing_text(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        accession = normalize_accession(accession)
        if accession is None:
            return ""

        cache_path = self.cache_dir / f"{accession}.txt"
        if cache_path.exists():
            return cache_path.read_text(
                encoding="utf-8", errors="ignore"
            )

        document = self.primary_document(
            cik, accession, supplied_document
        )
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        url = (
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/{document}"
        )
        raw_html = self.get_text(url)
        cleaned = clean_filing_html(raw_html)
        narrative = extract_narrative_sections(cleaned)
        cache_path.write_text(narrative, encoding="utf-8")
        return narrative


# Begin with any text already present in the experiment dataset.
model_data = model_data.copy()
model_data["sec_text"] = combine_available_text_columns(model_data)

# Merge a separate text table when configured or automatically found.
resolved_text_path = (
    TEXT_DATA_PATH
    if TEXT_DATA_PATH is not None
    else discover_text_data_path()
)
if resolved_text_path is not None:
    if not resolved_text_path.exists():
        raise FileNotFoundError(
            f"TEXT_DATA_PATH does not exist: {resolved_text_path}"
        )
    external_text = read_table(resolved_text_path)
    model_data = merge_external_text(model_data, external_text)
    print("Merged SEC text dataset:", resolved_text_path)

# Download only missing filing text, using one request per unique filing.
missing_text = model_data["sec_text"].fillna("").str.len() < MIN_TEXT_CHARS
can_download = (
    DOWNLOAD_SEC_TEXT_FROM_EDGAR
    and missing_text.any()
    and {"cik", "accession_number"}.issubset(model_data.columns)
    and valid_sec_user_agent(SEC_USER_AGENT)
)

if can_download:
    downloader = EdgarTextDownloader(
        SEC_USER_AGENT,
        TEXT_CACHE_DIR / "filings",
    )

    unique_filings = (
        model_data.loc[
            missing_text,
            [
                column
                for column in [
                    "cik",
                    "accession_number",
                    "primary_document",
                ]
                if column in model_data.columns
            ],
        ]
        .dropna(subset=["cik", "accession_number"])
        .drop_duplicates(subset=["cik", "accession_number"])
    )

    downloaded_text: dict[tuple[str, str], str] = {}
    failures = []

    for _, filing in tqdm(
        unique_filings.iterrows(),
        total=len(unique_filings),
        desc="Downloading SEC filings",
    ):
        cik = str(filing["cik"])
        accession = normalize_accession(filing["accession_number"])
        if accession is None:
            continue
        supplied_document = (
            filing.get("primary_document")
            if "primary_document" in filing.index
            else None
        )
        try:
            downloaded_text[(cik, accession)] = (
                downloader.filing_text(
                    cik,
                    accession,
                    supplied_document,
                )
            )
        except Exception as exc:
            failures.append(
                {
                    "cik": cik,
                    "accession_number": accession,
                    "error": str(exc),
                }
            )

    row_keys = list(
        zip(
            model_data["cik"].astype(str),
            model_data["accession_number"].map(normalize_accession),
        )
    )
    downloaded_series = pd.Series(
        [
            downloaded_text.get(key, "")
            for key in row_keys
        ],
        index=model_data.index,
    )
    replace_mask = (
        model_data["sec_text"].fillna("").str.len()
        < downloaded_series.str.len()
    )
    model_data.loc[replace_mask, "sec_text"] = downloaded_series[
        replace_mask
    ]

    if failures:
        failure_frame = pd.DataFrame(failures)
        failure_frame.to_csv(
            TEXT_CACHE_DIR / "edgar_download_failures.csv",
            index=False,
        )
        print(
            f"{len(failures)} filing downloads failed. Details were saved "
            "to edgar_download_failures.csv."
        )

elif missing_text.any():
    print(
        "SEC text is not yet available for all rows.\n"
        "Use one of these options:\n"
        "  1. Put sec_text, filing_text, mda_text, or risk_factors_text "
        "in the experiment dataset.\n"
        "  2. Set TEXT_DATA_PATH to a matching text parquet/CSV.\n"
        "  3. Replace SEC_USER_AGENT with your real name and email so "
        "the notebook can download filings from EDGAR."
    )

# Prevent obvious timing leakage when both timestamps are available.
if (
    "filing_date" in model_data.columns
    and "feature_cutoff_date" in model_data.columns
):
    filing_dates = pd.to_datetime(
        model_data["filing_date"], errors="coerce"
    )
    cutoff_dates = pd.to_datetime(
        model_data["feature_cutoff_date"], errors="coerce"
    )
    late_text = (
        filing_dates.notna()
        & cutoff_dates.notna()
        & (filing_dates > cutoff_dates)
    )
    if late_text.any():
        print(
            f"Removed text from {late_text.sum()} rows because the filing "
            "date was after the feature cutoff date."
        )
        model_data.loc[late_text, "sec_text"] = ""

model_data["text_characters"] = (
    model_data["sec_text"].fillna("").str.len()
)
model_data["has_sec_text"] = (
    model_data["text_characters"] >= MIN_TEXT_CHARS
)

text_coverage = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        rows_with_text=("has_sec_text", "sum"),
        median_text_characters=("text_characters", "median"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
text_coverage["coverage"] = (
    text_coverage["rows_with_text"] / text_coverage["rows"]
)
display(text_coverage)

## Keep identical text-available rows

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

text_model_data = model_data[
    model_data['has_sec_text'] & model_data['split'].isin(['train','validation','test'])
].copy()
for s in ['train','validation','test']:
    sub = text_model_data[text_model_data.split == s]
    if sub.empty or sub.target_clean.nunique() < 2:
        raise ValueError(f'{s} needs text rows from both classes. Add more text/data or reduce NEUTRAL_BAND.')
print(text_model_data.groupby('split').agg(rows=('target_clean','size'), companies=('cik','nunique'), classes=('target_clean','nunique')))
TEXT_MODELS_READY = True

# ------------------------------------------------------------------
# v7: LOCK ONE COMMON SET OF ROWS FOR ALL NUMERICAL + COMBINED MODELS
# ------------------------------------------------------------------
# The combined branch already requires usable SEC text.  v7 makes this
# restriction explicit and creates a stable row identifier so every model
# can be proven to have been evaluated on the same observations.

def make_observation_id(frame: pd.DataFrame) -> pd.Series:
    if {"cik", "quarter_end"}.issubset(frame.columns):
        quarter = pd.to_datetime(
            frame["quarter_end"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
        return (
            frame["cik"].astype(str).str.strip()
            + "__"
            + quarter.fillna("missing_date")
        )

    if {"ticker", "quarter_end"}.issubset(frame.columns):
        quarter = pd.to_datetime(
            frame["quarter_end"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
        return (
            frame["ticker"].astype(str).str.strip()
            + "__"
            + quarter.fillna("missing_date")
        )

    # Last-resort stable identifier inside this exported dataset.
    return pd.Series(
        [f"row_{i:06d}" for i in range(len(frame))],
        index=frame.index,
        dtype="object",
    )


text_model_data["observation_id"] = make_observation_id(
    text_model_data
)

duplicate_observation_ids = int(
    text_model_data["observation_id"].duplicated().sum()
)

if duplicate_observation_ids:
    raise ValueError(
        "Equal-row comparison cannot continue: "
        f"{duplicate_observation_ids} duplicate observation_id values found."
    )

MASTER_COMPARISON_ROWS = (
    text_model_data[
        [
            c for c in [
                "observation_id",
                "split",
                "cik",
                "ticker",
                "company_name",
                "quarter_end",
                "target_clean",
                "has_sec_text",
            ]
            if c in text_model_data.columns
        ]
    ]
    .sort_values(["split", "observation_id"])
    .reset_index(drop=True)
)

print("\nMASTER MATCHED-ROW MANIFEST")
print(
    MASTER_COMPARISON_ROWS.groupby("split")
    .size()
    .rename("rows")
)

print(
    "Total matched rows:",
    len(MASTER_COMPARISON_ROWS),
)

print(
    "Unique observation IDs:",
    MASTER_COMPARISON_ROWS["observation_id"].nunique(),
)

MASTER_COMPARISON_ROWS.to_csv(
    OUTPUT_DIR / "master_equal_row_manifest.csv",
    index=False,
)


## Sentence Transformer embeddings

### GPU acceleration (Kaggle T4 x2)

- Sentence Transformer encoding uses both CUDA devices when two GPUs are available.
- FinBERT creates one model copy per GPU and splits text chunks across the GPUs concurrently.
- GPU inference uses FP16 on CUDA for faster T4 Tensor Core execution.
- Sentence embeddings and FinBERT features are cached under `TEXT_CACHE_DIR` so reruns avoid repeated neural inference.
- TF-IDF, logistic regression, PCA, metrics, and threshold tuning remain CPU/scikit-learn operations.


In [ ]:
def chunk_document(
    text: str,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
    maximum_chunks: int = MAX_CHUNKS_PER_FILING,
) -> list[str]:
    words = str(text).split()
    if not words:
        return []

    step = max(1, chunk_words - overlap_words)
    chunks = [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if len(words[start:start + chunk_words]) >= 30
    ]

    if not chunks:
        return [" ".join(words)]

    if len(chunks) > maximum_chunks:
        selected_indices = np.linspace(
            0,
            len(chunks) - 1,
            maximum_chunks,
            dtype=int,
        )
        chunks = [chunks[index] for index in selected_indices]

    return chunks


def text_hash(text: str) -> str:
    payload = (
        TEXT_EMBEDDING_MODEL
        + "\n"
        + str(MAX_CHUNKS_PER_FILING)
        + "\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def get_torch_devices() -> list[str]:
    """Return all CUDA devices when available, otherwise CPU."""
    import torch

    if torch.cuda.is_available():
        return [
            f"cuda:{index}"
            for index in range(torch.cuda.device_count())
        ]
    return ["cpu"]


def build_sentence_embeddings(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from sentence_transformers import SentenceTransformer

    working = frame.copy()
    working["text_hash"] = working["sec_text"].map(text_hash)

    safe_model_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        TEXT_EMBEDDING_MODEL,
    )
    cache_path = (
        TEXT_CACHE_DIR
        / f"document_embeddings_{safe_model_name}.parquet"
    )

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["text_hash"])
        if not cached.empty and "text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        working[["text_hash", "sec_text"]]
        .drop_duplicates("text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["text_hash"].isin(cached_hashes)
    ]

    if not missing_documents.empty:
        devices = get_torch_devices()
        print("Sentence Transformer device(s):", devices)

        # Sentence Transformers supports a list such as
        # ["cuda:0", "cuda:1"] for multi-process multi-GPU encoding.
        encoder = SentenceTransformer(TEXT_EMBEDDING_MODEL)

        flat_chunks: list[str] = []
        owners: list[str] = []
        for row in missing_documents.itertuples(index=False):
            chunks = chunk_document(row.sec_text)
            flat_chunks.extend(chunks)
            owners.extend([row.text_hash] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid text chunks were produced.")

        encode_device = devices if len(devices) > 1 else devices[0]

        try:
            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=encode_device,
            )
        except Exception as exc:
            # Multi-process GPU startup can occasionally fail in a notebook
            # runtime. Fall back to the first GPU rather than losing the run.
            if len(devices) <= 1:
                raise
            print(
                "Multi-GPU Sentence Transformer failed; "
                "falling back to cuda:0. Error:",
                repr(exc),
            )
            encoder = SentenceTransformer(
                TEXT_EMBEDDING_MODEL,
                device=devices[0],
            )
            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=devices[0],
            )

        chunk_frame = pd.DataFrame(chunk_embeddings)
        chunk_frame.insert(0, "text_hash", owners)

        document_embeddings = (
            chunk_frame.groupby("text_hash", sort=False)
            .mean()
            .reset_index()
        )

        embedding_columns = [
            column
            for column in document_embeddings.columns
            if column != "text_hash"
        ]
        matrix = document_embeddings[embedding_columns].to_numpy()
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        matrix = matrix / np.maximum(norms, 1e-12)
        document_embeddings[embedding_columns] = matrix

        if cached.empty:
            cached = document_embeddings
        else:
            cached = pd.concat(
                [cached, document_embeddings],
                ignore_index=True,
            ).drop_duplicates("text_hash", keep="last")

        cached.to_parquet(cache_path, index=False)

    embedding_columns_in_cache = [
        column for column in cached.columns
        if column != "text_hash"
    ]
    renamed = {
        column: f"text_embedding_{int(column):03d}"
        for column in embedding_columns_in_cache
    }
    cached = cached.rename(columns=renamed)
    embedding_columns = list(renamed.values())

    working = working.merge(
        cached,
        on="text_hash",
        how="left",
    )
    return working, embedding_columns


if TEXT_MODELS_READY:
    text_model_data, text_embedding_columns = (
        build_sentence_embeddings(text_model_data)
    )
    print(
        "Sentence embedding dimensions:",
        len(text_embedding_columns),
    )
else:
    text_embedding_columns = []

## FinBERT features

In [ ]:
def finbert_text_hash(text: str) -> str:
    payload = (
        FINBERT_MODEL_NAME
        + "\n180\n30\n12\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def _finbert_infer_chunks(
    chunks: list[str],
    indices: np.ndarray,
    device: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Run one independent FinBERT worker on one GPU."""
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_NAME)

    model_kwargs = {}
    if device.startswith("cuda"):
        # T4 Tensor Cores are much faster with FP16 inference.
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL_NAME,
        **model_kwargs,
    )
    model.to(device)
    model.eval()

    probability_batches = []
    for start in range(0, len(indices), TEXT_BATCH_SIZE):
        batch_indices = indices[start:start + TEXT_BATCH_SIZE]
        batch = [chunks[int(i)] for i in batch_indices]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            if device.startswith("cuda"):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = model(**tokens).logits
            else:
                logits = model(**tokens).logits

            batch_probabilities = torch.softmax(
                logits,
                dim=-1,
            ).float().cpu().numpy()

        probability_batches.append(batch_probabilities)

    if not probability_batches:
        return indices, np.empty((0, model.config.num_labels))

    return indices, np.vstack(probability_batches)


def build_finbert_sentiment_features(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from transformers import AutoConfig

    output = frame.copy()
    feature_names = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]

    if not RUN_FINBERT_SENTIMENT:
        for feature in feature_names:
            output[feature] = 0.0
        return output, []

    output["finbert_text_hash"] = output["sec_text"].map(
        finbert_text_hash
    )
    cache_path = TEXT_CACHE_DIR / "finbert_sentiment_features.parquet"

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["finbert_text_hash"])
        if not cached.empty and "finbert_text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        output[["finbert_text_hash", "sec_text"]]
        .drop_duplicates("finbert_text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["finbert_text_hash"].isin(cached_hashes)
    ].reset_index(drop=True)

    if not missing_documents.empty:
        devices = get_torch_devices()
        print("FinBERT device(s):", devices)

        config = AutoConfig.from_pretrained(FINBERT_MODEL_NAME)
        id_to_label = {
            int(index): str(label).lower()
            for index, label in config.id2label.items()
        }
        num_labels = len(id_to_label)

        flat_chunks: list[str] = []
        owners: list[int] = []

        for doc_index, row in enumerate(
            missing_documents.itertuples(index=False)
        ):
            chunks = chunk_document(
                row.sec_text,
                chunk_words=180,
                overlap_words=30,
                maximum_chunks=12,
            )
            if not chunks:
                continue
            flat_chunks.extend(chunks)
            owners.extend([doc_index] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid FinBERT text chunks were produced.")

        all_indices = np.arange(len(flat_chunks), dtype=int)
        index_splits = [
            split.astype(int)
            for split in np.array_split(all_indices, len(devices))
            if len(split) > 0
        ]

        probability_matrix = np.empty(
            (len(flat_chunks), num_labels),
            dtype=np.float32,
        )

        if len(index_splits) == 1:
            indices, probs = _finbert_infer_chunks(
                flat_chunks,
                index_splits[0],
                devices[0],
            )
            probability_matrix[indices] = probs
        else:
            # One independent model copy per T4. Each GPU processes a
            # different half of the chunks concurrently.
            with ThreadPoolExecutor(
                max_workers=len(index_splits)
            ) as executor:
                futures = []
                for worker_index, indices in enumerate(index_splits):
                    device = devices[worker_index]
                    futures.append(
                        executor.submit(
                            _finbert_infer_chunks,
                            flat_chunks,
                            indices,
                            device,
                        )
                    )

                for future in as_completed(futures):
                    indices, probs = future.result()
                    probability_matrix[indices] = probs

        owners_array = np.asarray(owners, dtype=int)
        feature_rows = []

        for doc_index, row in enumerate(
            missing_documents.itertuples(index=False)
        ):
            doc_mask = owners_array == doc_index
            matrix = probability_matrix[doc_mask]

            if len(matrix) == 0:
                positive = np.array([0.0])
                neutral = np.array([0.0])
                negative = np.array([0.0])
            else:
                label_columns = {
                    label: matrix[:, index]
                    for index, label in id_to_label.items()
                }
                positive = label_columns.get(
                    "positive",
                    np.zeros(len(matrix)),
                )
                neutral = label_columns.get(
                    "neutral",
                    np.zeros(len(matrix)),
                )
                negative = label_columns.get(
                    "negative",
                    np.zeros(len(matrix)),
                )

            feature_rows.append(
                {
                    "finbert_text_hash": row.finbert_text_hash,
                    "finbert_positive_mean": float(positive.mean()),
                    "finbert_neutral_mean": float(neutral.mean()),
                    "finbert_negative_mean": float(negative.mean()),
                    "finbert_negative_max": float(negative.max()),
                    "finbert_negative_std": float(negative.std()),
                    "finbert_positive_minus_negative": float(
                        positive.mean() - negative.mean()
                    ),
                }
            )

        new_features = pd.DataFrame(feature_rows)
        if cached.empty:
            cached = new_features
        else:
            cached = pd.concat(
                [cached, new_features],
                ignore_index=True,
            ).drop_duplicates(
                "finbert_text_hash",
                keep="last",
            )

        cached.to_parquet(cache_path, index=False)

    output = output.merge(
        cached[["finbert_text_hash"] + feature_names],
        on="finbert_text_hash",
        how="left",
    )

    return output, feature_names


if TEXT_MODELS_READY:
    text_model_data, finbert_feature_columns = (
        build_finbert_sentiment_features(text_model_data)
    )
else:
    finbert_feature_columns = []

print("FinBERT features included:", finbert_feature_columns)

## Filing-language change features

This experiment augments the **level** of current filing language with the
quarter-to-quarter change in that language.

For sentence embeddings:

\[
\Delta E_t = E_t - E_{t-1}
\]

For FinBERT sentiment:

\[
\Delta S_t = S_t - S_{t-1}
\]

These are the textual analogue of financial momentum/change features.

In [ ]:
def add_filing_language_change_features(frame):
    """Add quarter-over-quarter changes for numeric text features.

    Deltas are created BEFORE model fitting in v5 so they can actually be
    selected and evaluated by the main models.
    """
    result = frame.copy()

    company_col = next(
        (c for c in ["cik", "ticker", "company_name"] if c in result.columns),
        None,
    )
    date_col = next(
        (
            c for c in [
                "feature_cutoff_date",
                "source_filing_date",
                "filing_date",
                "quarter_end",
            ]
            if c in result.columns
        ),
        None,
    )
    if company_col is None or date_col is None:
        raise ValueError("Company/date columns are required for language-change features.")

    result[date_col] = pd.to_datetime(result[date_col], errors="coerce")
    result = result.sort_values([company_col, date_col]).copy()

    embedding_level_columns = [
        c for c in result.columns if c.startswith("text_embedding_")
    ]
    known_finbert_features = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]
    finbert_level_columns = [
        c for c in known_finbert_features if c in result.columns
    ]

    level_columns = embedding_level_columns + finbert_level_columns
    if not level_columns:
        print("No numeric Sentence Transformer / FinBERT level features found.")
        return result

    for column in level_columns:
        result[column] = pd.to_numeric(result[column], errors="coerce")
        result[f"{column}_delta1"] = (
            result.groupby(company_col, dropna=False)[column].diff()
        )

    print(
        "Language-change source features:",
        len(level_columns),
        f"({len(embedding_level_columns)} embeddings + "
        f"{len(finbert_level_columns)} FinBERT)",
    )
    return result

text_model_data = add_filing_language_change_features(text_model_data)

embedding_delta_columns = [
    c for c in text_model_data.columns
    if c.startswith("text_embedding_") and c.endswith("_delta1")
]
finbert_delta_columns = [
    c for c in text_model_data.columns
    if c.startswith("finbert_") and c.endswith("_delta1")
]

print("Sentence embedding delta features:", len(embedding_delta_columns))
print("FinBERT delta features:", len(finbert_delta_columns))


## Pre-fit data and text audit


In [ ]:
# v5 data / text audit before model fitting
audit_rows = []
for split_name in ["train", "validation", "test"]:
    part = text_model_data[text_model_data["split"] == split_name]
    audit_rows.append({
        "split": split_name,
        "rows": len(part),
        "companies": part["cik"].nunique() if "cik" in part.columns else np.nan,
        "acceleration_rate": float(part["target_clean"].mean()) if len(part) else np.nan,
        "median_text_chars": float(part["sec_text"].astype(str).str.len().median()) if len(part) else np.nan,
    })

data_audit = pd.DataFrame(audit_rows)
display(data_audit)

duplicate_keys = [
    c for c in ["cik", "quarter_end"] if c in text_model_data.columns
]
if len(duplicate_keys) == 2:
    duplicate_count = int(text_model_data.duplicated(duplicate_keys).sum())
    print("Duplicate company-quarter rows:", duplicate_count)

if {"source_filing_date", "feature_cutoff_date"}.issubset(text_model_data.columns):
    source_date = pd.to_datetime(text_model_data["source_filing_date"], errors="coerce")
    cutoff_date = pd.to_datetime(text_model_data["feature_cutoff_date"], errors="coerce")
    future_text_rows = int((source_date > cutoff_date).fillna(False).sum())
    print("Rows where source filing date is AFTER feature cutoff:", future_text_rows)


## Train combined numerical + textual models

In [ ]:
def choose_threshold(y_true, probabilities):
    scored = []
    for threshold in THRESHOLD_GRID:
        pred = (probabilities >= threshold).astype(int)
        scored.append((
            float(threshold),
            float(balanced_accuracy_score(y_true, pred)),
        ))
    best_score = max(score for _, score in scored)
    tied = [
        item for item in scored
        if np.isclose(item[1], best_score, rtol=0.0, atol=1e-12)
    ]
    return min(tied, key=lambda item: abs(item[0] - 0.50))


def evaluation_row(name, y, p, pred, base_p):
    base = np.full(len(y), base_p)
    base_brier = brier_score_loss(y, base)
    brier = brier_score_loss(y, p)
    return {
        "model": name,
        "rows": len(y),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "acceleration_precision": precision_score(y, pred, pos_label=1, zero_division=0),
        "acceleration_recall": recall_score(y, pred, pos_label=1, zero_division=0),
        "deceleration_recall": recall_score(y, pred, pos_label=0, zero_division=0),
        "brier_score": brier,
        "brier_skill_score": 1 - brier / base_brier if base_brier > 0 else np.nan,
        "roc_auc": roc_auc_score(y, p) if pd.Series(y).nunique() == 2 else np.nan,
        "average_precision": average_precision_score(y, p) if pd.Series(y).nunique() == 2 else np.nan,
    }


def is_better_candidate(candidate, best):
    if best is None:
        return True
    if candidate["val_bal"] > best["val_bal"] + 1e-12:
        return True
    if np.isclose(candidate["val_bal"], best["val_bal"], rtol=0.0, atol=1e-12):
        if candidate["brier"] < best["brier"] - 1e-12:
            return True
    return False


def make_combined_pipeline(
    mode,
    C,
    class_weight,
    train_rows,
    min_df,
    pca_components=32,
):
    branches = []

    # This branch is included in EVERY model, including the explicit
    # numerical-only same-row control.
    if numeric_features:
        branches.append((
            "financial_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ))

    if categorical_features:
        branches.append((
            "financial_categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ))

    if mode == "tfidf":
        branches.append((
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                stop_words="english",
                ngram_range=(1, 2),
                min_df=min_df,
                max_df=0.98,
                max_features=20000,
                sublinear_tf=True,
            ),
            "sec_text",
        ))

    use_sentence = mode in {
        "sentence", "sentence_delta",
        "sentence_finbert", "sentence_finbert_delta",
    }
    use_finbert = mode in {
        "finbert", "finbert_delta",
        "sentence_finbert", "sentence_finbert_delta",
    }
    use_delta = mode.endswith("_delta")

    if use_sentence:
        sentence_features = list(text_embedding_columns)
        if use_delta:
            sentence_features += list(embedding_delta_columns)

        ncomp = max(
            1,
            min(int(pca_components), len(sentence_features), train_rows - 1),
        )
        branches.append((
            "sentence_embeddings",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=ncomp, random_state=RANDOM_STATE)),
            ]),
            sentence_features,
        ))

    if use_finbert:
        fin_features = list(finbert_feature_columns)
        if use_delta:
            fin_features += list(finbert_delta_columns)

        branches.append((
            "finbert",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            fin_features,
        ))

    prep = ColumnTransformer(
        branches,
        remainder="drop",
        verbose_feature_names_out=False,
    )
    solver = "liblinear" if mode == "tfidf" else "lbfgs"
    return Pipeline([
        ("preprocessor", prep),
        ("classifier", LogisticRegression(
            C=C,
            penalty="l2",
            class_weight=class_weight,
            solver=solver,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])


train = text_model_data[text_model_data.split == "train"].copy()
val = text_model_data[text_model_data.split == "validation"].copy()
test = text_model_data[text_model_data.split == "test"].copy()
y_train = train.target_clean.astype(int)
y_val = val.target_clean.astype(int)
y_test = test.target_clean.astype(int)

# v7 comparison-integrity checks.
expected_ids = {
    split_name: set(
        MASTER_COMPARISON_ROWS.loc[
            MASTER_COMPARISON_ROWS["split"].eq(split_name),
            "observation_id",
        ]
    )
    for split_name in ["train", "validation", "test"]
}

actual_frames = {
    "train": train,
    "validation": val,
    "test": test,
}

for split_name, split_frame in actual_frames.items():
    actual_ids = set(split_frame["observation_id"])
    if actual_ids != expected_ids[split_name]:
        raise AssertionError(
            f"{split_name}: model rows do not match the locked master manifest."
        )

print("\nEqual-row assertions passed.")
print(
    "Train / validation / test rows:",
    len(train), len(val), len(test),
)


base_p = float(y_train.mean())
min_df = 1 if len(train) < 100 else 2

# IMPORTANT: "Numerical only" is evaluated on EXACTLY the same text-available
# train/validation/test rows as every fused model below. This makes the
# incremental contribution of text directly measurable.
specs = {
    "Numerical only + Logistic Regression": "numerical",
    "Numerical + TF-IDF + Logistic Regression": "tfidf",
    "Numerical + Sentence Transformer + Logistic Regression": "sentence",
    "Numerical + Sentence Transformer + Delta + Logistic Regression": "sentence_delta",
    "Numerical + FinBERT + Logistic Regression": "finbert",
    "Numerical + FinBERT + Delta + Logistic Regression": "finbert_delta",
    "Numerical + Sentence Transformer + FinBERT + Logistic Regression": "sentence_finbert",
    "Numerical + Sentence Transformer + FinBERT + Delta + Logistic Regression": "sentence_finbert_delta",
}

rows = []
selection = []
models = {}
preds = test[[
    c for c in [
        "observation_id", "cik", "ticker", "company_name", "quarter_end",
        "future_growth_change", "target_clean",
    ]
    if c in test.columns
]].copy()

prior = np.full(len(test), base_p)
rows.append(evaluation_row(
    "Prior-probability baseline",
    y_test,
    prior,
    (prior >= 0.5).astype(int),
    base_p,
))

for name, mode in specs.items():
    best = None
    uses_sentence = "sentence" in mode
    pca_grid = TEXT_PCA_GRID if uses_sentence else [1]

    for pca_components in pca_grid:
        for w in [None, "balanced"]:
            for C in C_GRID:
                pipe = make_combined_pipeline(
                    mode,
                    C,
                    w,
                    len(train),
                    min_df,
                    pca_components=pca_components,
                )
                pipe.fit(train, y_train)
                val_p = pipe.predict_proba(val)[:, 1]
                threshold, val_bal = choose_threshold(y_val, val_p)

                candidate = {
                    "C": C,
                    "class_weight": w,
                    "threshold": float(threshold),
                    "val_bal": float(val_bal),
                    "brier": float(brier_score_loss(y_val, val_p)),
                    "pca_components": int(pca_components) if uses_sentence else np.nan,
                }
                if is_better_candidate(candidate, best):
                    best = candidate

    final = make_combined_pipeline(
        mode,
        best["C"],
        best["class_weight"],
        len(train),
        min_df,
        pca_components=best["pca_components"] if uses_sentence else 1,
    )

    # Keep TRAIN-only fitting because validation chose the threshold for a
    # train-fitted model. This avoids the old stale-threshold-after-refit bug.
    final.fit(train, y_train)

    test_p = final.predict_proba(test)[:, 1]
    test_pred = (test_p >= best["threshold"]).astype(int)

    row = evaluation_row(name, y_test, test_p, test_pred, base_p)
    row.update({
        "selected_C": best["C"],
        "class_weight": str(best["class_weight"]),
        "threshold": best["threshold"],
        "validation_balanced_accuracy": best["val_bal"],
        "validation_brier": best["brier"],
        "selected_pca_components": best["pca_components"],
        "selection_metric": "balanced_accuracy",
    })
    rows.append(row)
    selection.append({"model": name, **best})
    models[name] = final
    preds[f"{name}_probability"] = test_p
    preds[f"{name}_prediction"] = test_pred

results = pd.DataFrame(rows).sort_values(
    ["balanced_accuracy", "brier_score"],
    ascending=[False, True],
)

# Explicit incremental-text diagnostic versus the same-row numerical control.
numerical_mask = results["model"].eq("Numerical only + Logistic Regression")
if numerical_mask.any():
    numerical_ba = float(results.loc[numerical_mask, "balanced_accuracy"].iloc[0])
    results["balanced_accuracy_delta_vs_same_row_numerical"] = (
        results["balanced_accuracy"] - numerical_ba
    )

selection = pd.DataFrame(selection).sort_values(
    ["val_bal", "brier"],
    ascending=[False, True],
)

display(results)
display(selection)

results.to_csv(OUTPUT_DIR / "combined_test_results.csv", index=False)
selection.to_csv(OUTPUT_DIR / "combined_validation_selection.csv", index=False)
preds.to_csv(OUTPUT_DIR / "combined_test_predictions.csv", index=False)

with pd.ExcelWriter(
    OUTPUT_DIR / "combined_branch_results.xlsx",
    engine="openpyxl",
) as writer:
    results.to_excel(writer, sheet_name="Model_Results", index=False)
    selection.to_excel(writer, sheet_name="Validation_Selection", index=False)
    preds.to_excel(writer, sheet_name="Test_Predictions", index=False)
    data_audit.to_excel(writer, sheet_name="Data_Audit", index=False)

joblib.dump(models, OUTPUT_DIR / "combined_final_models.joblib")
print("Saved to:", OUTPUT_DIR)


## v6 — Random Forest and XGBoost multimodal fusion


In [ ]:
# v6: Random Forest and XGBoost multimodal fusion
# Candidate grids are the same numerical model families/settings used in Branch 1.
RF_PARAM_GRID = [
    {'n_estimators':300,'max_depth':None,'min_samples_leaf':2,'max_features':'sqrt'},
    {'n_estimators':500,'max_depth':6,'min_samples_leaf':2,'max_features':'sqrt'},
    {'n_estimators':500,'max_depth':10,'min_samples_leaf':3,'max_features':0.7},
]
XGB_PARAM_GRID = [
    {'n_estimators':200,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
    {'n_estimators':300,'max_depth':3,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
    {'n_estimators':300,'max_depth':4,'learning_rate':0.02,'subsample':0.8,'colsample_bytree':0.8},
]

def make_tree_preprocessor(mode, train_rows, pca_components=32):
    branches=[]
    if numeric_features:
        branches.append(('financial_numeric', Pipeline([
            ('imputer',SimpleImputer(strategy='median'))
        ]), numeric_features))
    if categorical_features:
        branches.append(('financial_categorical', Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features))

    use_sentence = mode in {'sentence','sentence_delta','sentence_finbert','sentence_finbert_delta'}
    use_finbert = mode in {'finbert','finbert_delta','sentence_finbert','sentence_finbert_delta'}
    use_delta = mode.endswith('_delta')

    if use_sentence:
        cols=list(text_embedding_columns)
        if use_delta: cols += list(embedding_delta_columns)
        ncomp=max(1,min(int(pca_components),len(cols),train_rows-1))
        branches.append(('sentence_embeddings', Pipeline([
            ('imputer',SimpleImputer(strategy='constant',fill_value=0.0)),
            ('scaler',StandardScaler()),
            ('pca',PCA(n_components=ncomp,random_state=RANDOM_STATE))
        ]), cols))

    if use_finbert:
        cols=list(finbert_feature_columns)
        if use_delta: cols += list(finbert_delta_columns)
        branches.append(('finbert', Pipeline([
            ('imputer',SimpleImputer(strategy='median'))
        ]), cols))

    return ColumnTransformer(branches,remainder='drop',verbose_feature_names_out=False)

def build_rf(mode, params, pca_components=32):
    return Pipeline([
        ('preprocessor',make_tree_preprocessor(mode,len(train),pca_components)),
        ('classifier',RandomForestClassifier(**params,class_weight='balanced_subsample',random_state=RANDOM_STATE,n_jobs=-1))
    ])

def build_xgb(mode, params, pca_components=32):
    pos=max(1,int((y_train==1).sum())); neg=max(1,int((y_train==0).sum()))
    extra={'tree_method':'hist'}
    try:
        import torch
        if torch.cuda.is_available(): extra['device']='cuda'
    except Exception:
        pass
    return Pipeline([
        ('preprocessor',make_tree_preprocessor(mode,len(train),pca_components)),
        ('classifier',XGBClassifier(**params,scale_pos_weight=neg/pos,objective='binary:logistic',eval_metric='logloss',random_state=RANDOM_STATE,n_jobs=-1,**extra))
    ])

def select_best_numeric_family(family, grid):
    best=None
    print('\nSelecting same-row numerical',family)
    for params in grid:
        model = build_rf('numerical',params,1) if family=='Random Forest' else build_xgb('numerical',params,1)
        model.fit(train,y_train)
        vp=model.predict_proba(val)[:,1]
        threshold,val_bal=choose_threshold(y_val,vp)
        cand={'params':params,'threshold':float(threshold),'val_bal':float(val_bal),'brier':float(brier_score_loss(y_val,vp))}
        print(params,'val BA=',round(cand['val_bal'],4))
        if is_better_candidate(cand,best): best=cand
    print('BEST',family,best)
    return best

best_rf=select_best_numeric_family('Random Forest',RF_PARAM_GRID)
best_xgb=select_best_numeric_family('XGBoost',XGB_PARAM_GRID)

def run_tree_fusion(family,best_numeric):
    modes=[
        ('Numerical only','numerical'),
        ('Numerical + Sentence Transformer','sentence'),
        ('Numerical + Sentence Transformer + Delta','sentence_delta'),
        ('Numerical + FinBERT','finbert'),
        ('Numerical + FinBERT + Delta','finbert_delta'),
        ('Numerical + Sentence Transformer + FinBERT','sentence_finbert'),
        ('Numerical + Sentence Transformer + FinBERT + Delta','sentence_finbert_delta'),
    ]
    out=[]; sel=[]; fitted={}
    for label,mode in modes:
        best=None
        pca_grid=TEXT_PCA_GRID if 'sentence' in mode else [1]
        print('\n',family,'|',label)
        for pca in pca_grid:
            model = build_rf(mode,best_numeric['params'],pca) if family=='Random Forest' else build_xgb(mode,best_numeric['params'],pca)
            model.fit(train,y_train)
            vp=model.predict_proba(val)[:,1]
            threshold,val_bal=choose_threshold(y_val,vp)
            cand={'params':best_numeric['params'],'threshold':float(threshold),'val_bal':float(val_bal),'brier':float(brier_score_loss(y_val,vp)),'pca_components':int(pca) if 'sentence' in mode else np.nan}
            if is_better_candidate(cand,best): best=cand
        final = build_rf(mode,best['params'],best['pca_components'] if 'sentence' in mode else 1) if family=='Random Forest' else build_xgb(mode,best['params'],best['pca_components'] if 'sentence' in mode else 1)
        final.fit(train,y_train)
        p=final.predict_proba(test)[:,1]; pred=(p>=best['threshold']).astype(int)
        name=f'{label} + {family}'
        row=evaluation_row(name,y_test,p,pred,base_p)
        row.update({'model_family':family,'selected_params':str(best['params']),'selected_threshold':best['threshold'],'validation_balanced_accuracy':best['val_bal'],'validation_brier':best['brier'],'selected_pca_components':best['pca_components']})
        out.append(row); sel.append({'model':name,**best}); fitted[name]=final
        preds[f'{name}_probability']=p; preds[f'{name}_prediction']=pred
        print('selected val BA=',round(best['val_bal'],4),'test BA=',round(row['balanced_accuracy'],4))
    df=pd.DataFrame(out)
    base=df.loc[df['model'].eq(f'Numerical only + {family}'),'balanced_accuracy']
    if not base.empty:
        df['balanced_accuracy_delta_vs_same_row_numerical']=df['balanced_accuracy']-float(base.iloc[0])
    return df,pd.DataFrame(sel),fitted

rf_results,rf_selection,rf_models=run_tree_fusion('Random Forest',best_rf)
xgb_results,xgb_selection,xgb_models=run_tree_fusion('XGBoost',best_xgb)

all_combined_results=pd.concat([
    results.assign(experiment_family='Logistic Regression control'),
    rf_results.assign(experiment_family='Random Forest'),
    xgb_results.assign(experiment_family='XGBoost')
],ignore_index=True,sort=False).sort_values(['balanced_accuracy','brier_score'],ascending=[False,True]).reset_index(drop=True)

tree_selection=pd.concat([
    rf_selection.assign(model_family='Random Forest'),
    xgb_selection.assign(model_family='XGBoost')
],ignore_index=True,sort=False)
models.update(rf_models); models.update(xgb_models)

display(all_combined_results)
display(tree_selection)
all_combined_results.to_csv(OUTPUT_DIR/'all_combined_model_results_v6.csv',index=False)
tree_selection.to_csv(OUTPUT_DIR/'tree_model_validation_selection_v6.csv',index=False)
preds.to_csv(OUTPUT_DIR/'combined_test_predictions_v6.csv',index=False)
with pd.ExcelWriter(OUTPUT_DIR/'combined_branch_v6_results.xlsx',engine='openpyxl') as writer:
    all_combined_results.to_excel(writer,sheet_name='All_Model_Results',index=False)
    tree_selection.to_excel(writer,sheet_name='Tree_Validation_Selection',index=False)
    selection.to_excel(writer,sheet_name='LR_Validation_Selection',index=False)
    data_audit.to_excel(writer,sheet_name='Data_Audit',index=False)
joblib.dump(models,OUTPUT_DIR/'combined_v6_final_models.joblib')
print('Saved v6 results to:',OUTPUT_DIR)


## v8 validation-selected best numerical and combined systems


In [ ]:
paper_selection_rows = []
for family in ["Random Forest","XGBoost"]:
    family_sel = tree_selection[
        tree_selection["model_family"].eq(family)
    ].copy()
    if family_sel.empty:
        continue

    num = family_sel[
        family_sel["model"].str.startswith("Numerical only",na=False)
    ].copy()
    comb = family_sel[
        ~family_sel["model"].str.startswith("Numerical only",na=False)
    ].copy()

    if not num.empty:
        r = num.sort_values(["val_bal","brier"],ascending=[False,True]).iloc[0]
        paper_selection_rows.append({
            "model_family":family,
            "comparison_role":"best_numerical_validation_selected",
            "model":r["model"],
            "validation_balanced_accuracy":r["val_bal"],
            "validation_brier":r["brier"],
        })
    if not comb.empty:
        r = comb.sort_values(["val_bal","brier"],ascending=[False,True]).iloc[0]
        paper_selection_rows.append({
            "model_family":family,
            "comparison_role":"best_combined_validation_selected",
            "model":r["model"],
            "validation_balanced_accuracy":r["val_bal"],
            "validation_brier":r["brier"],
        })

independent_validation_selection = pd.DataFrame(paper_selection_rows)
display(independent_validation_selection)
independent_validation_selection.to_csv(
    OUTPUT_DIR/"independent_validation_selection_v8.csv",index=False
)


## v8.1 paired company-cluster bootstrap + McNemar + Holm correction

The company-cluster bootstrap remains the **primary** inference for the balanced-accuracy gap.

McNemar remains a **secondary** test for overall accuracy. Because multiple numerical-vs-combined McNemar tests are run, the raw p-values are adjusted using the **Holm step-down procedure** to control the family-wise error rate.

Report both:
- `mcnemar_exact_p_value` — raw p-value
- `mcnemar_holm_adjusted_p_value` — multiplicity-adjusted p-value
- `mcnemar_holm_reject_0_05` — whether the comparison remains significant after Holm correction at α = 0.05


In [ ]:
from scipy.stats import binomtest

def holm_adjust_pvalues(p_values, alpha=0.05):
    p = np.asarray(p_values, dtype=float)
    if p.ndim != 1:
        raise ValueError("p_values must be one-dimensional")
    m = len(p)
    if m == 0:
        return np.array([], dtype=float), np.array([], dtype=bool)
    if np.any(~np.isfinite(p)):
        raise ValueError("All p-values must be finite")
    order = np.argsort(p)
    sorted_p = p[order]
    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0
    for j, p_j in enumerate(sorted_p):
        running_max = max(running_max, (m-j) * p_j)
        adjusted_sorted[j] = min(1.0, running_max)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted
    reject = adjusted <= alpha
    return adjusted, reject


BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 2027

company_id_column = next(
    (c for c in ["cik","ticker","company_name"] if c in preds.columns),
    None,
)
if company_id_column is None:
    raise ValueError("Need a company identifier for clustered bootstrap.")

def company_cluster_bootstrap(frame,num_col,comb_col,iterations=5000,seed=2027):
    rng = np.random.default_rng(seed)
    companies = frame[company_id_column].dropna().unique()
    blocks = {
        c:frame.index[frame[company_id_column].eq(c)].to_numpy()
        for c in companies
    }
    ba_delta=[]
    acc_delta=[]
    attempts=0
    while len(ba_delta)<iterations and attempts<iterations*20:
        attempts += 1
        sampled = rng.choice(companies,size=len(companies),replace=True)
        idx = np.concatenate([blocks[c] for c in sampled])
        sample = frame.loc[idx]
        y = sample["target_clean"].astype(int).to_numpy()
        if np.unique(y).size<2:
            continue
        n = sample[num_col].astype(int).to_numpy()
        c = sample[comb_col].astype(int).to_numpy()
        ba_delta.append(
            balanced_accuracy_score(y,c)-balanced_accuracy_score(y,n)
        )
        acc_delta.append(
            accuracy_score(y,c)-accuracy_score(y,n)
        )
    return np.asarray(ba_delta),np.asarray(acc_delta)

def exact_mcnemar(y_true,num_pred,comb_pred):
    y=np.asarray(y_true,dtype=int)
    n=np.asarray(num_pred,dtype=int)
    c=np.asarray(comb_pred,dtype=int)
    nc=n==y
    cc=c==y
    num_only=int(np.sum(nc & ~cc))
    comb_only=int(np.sum(~nc & cc))
    discordant=num_only+comb_only
    p=1.0 if discordant==0 else float(
        binomtest(
            min(num_only,comb_only),
            n=discordant,p=0.5,alternative="two-sided"
        ).pvalue
    )
    return num_only,comb_only,discordant,p

def add_pair(rows,family,num_name,comb_name):
    num_col=f"{num_name}_prediction"
    comb_col=f"{comb_name}_prediction"
    if num_col not in preds.columns or comb_col not in preds.columns:
        return

    locked=set(MASTER_COMPARISON_ROWS.loc[
        MASTER_COMPARISON_ROWS["split"].eq("test"),"observation_id"
    ])
    if set(preds["observation_id"]) != locked:
        raise AssertionError("Paired predictions do not match locked test rows.")

    y=preds["target_clean"].astype(int).to_numpy()
    n=preds[num_col].astype(int).to_numpy()
    c=preds[comb_col].astype(int).to_numpy()

    ba_obs=balanced_accuracy_score(y,c)-balanced_accuracy_score(y,n)
    acc_obs=accuracy_score(y,c)-accuracy_score(y,n)

    ba_boot,acc_boot=company_cluster_bootstrap(
        preds,num_col,comb_col,
        iterations=BOOTSTRAP_ITERATIONS,seed=BOOTSTRAP_SEED
    )
    num_only,comb_only,discordant,p=exact_mcnemar(y,n,c)

    rows.append({
        "model_family":family,
        "numerical_model":num_name,
        "combined_model":comb_name,
        "n_test_rows":len(preds),
        "n_test_companies":int(preds[company_id_column].nunique()),
        "observed_balanced_accuracy_gap":ba_obs,
        "cluster_bootstrap_ba_ci95_low":float(np.quantile(ba_boot,0.025)),
        "cluster_bootstrap_ba_ci95_high":float(np.quantile(ba_boot,0.975)),
        "cluster_bootstrap_fraction_ba_gt_zero":float(np.mean(ba_boot>0)),
        "observed_accuracy_gap":acc_obs,
        "cluster_bootstrap_accuracy_ci95_low":float(np.quantile(acc_boot,0.025)),
        "cluster_bootstrap_accuracy_ci95_high":float(np.quantile(acc_boot,0.975)),
        "numerical_only_correct":num_only,
        "combined_only_correct":comb_only,
        "discordant_pairs":discordant,
        "mcnemar_exact_p_value":p,
        "mcnemar_note":"Supplementary: repeated quarters violate ordinary pair-independence.",
    })

comparison_specs={
    "Logistic Regression":(
        "Numerical only + Logistic Regression",
        [
            "Numerical + TF-IDF + Logistic Regression",
            "Numerical + Sentence Transformer + Logistic Regression",
            "Numerical + FinBERT + Logistic Regression",
            "Numerical + Sentence Transformer + FinBERT + Logistic Regression",
        ],
    ),
    "Random Forest":(
        "Numerical only + Random Forest",
        [
            "Numerical + Sentence Transformer + Random Forest",
            "Numerical + FinBERT + Random Forest",
            "Numerical + Sentence Transformer + FinBERT + Random Forest",
        ],
    ),
    "XGBoost":(
        "Numerical only + XGBoost",
        [
            "Numerical + Sentence Transformer + XGBoost",
            "Numerical + FinBERT + XGBoost",
            "Numerical + Sentence Transformer + FinBERT + XGBoost",
        ],
    ),
}

paired_inference_rows=[]
for family,(num_name,comb_names) in comparison_specs.items():
    for comb_name in comb_names:
        add_pair(paired_inference_rows,family,num_name,comb_name)

paired_inference_results=pd.DataFrame(paired_inference_rows)
if not paired_inference_results.empty:
    holm_adjusted, holm_reject = holm_adjust_pvalues(
        paired_inference_results["mcnemar_exact_p_value"].to_numpy(),
        alpha=0.05,
    )
    paired_inference_results["mcnemar_holm_adjusted_p_value"] = holm_adjusted
    paired_inference_results["mcnemar_holm_reject_0_05"] = holm_reject
    paired_inference_results["mcnemar_raw_reject_0_05"] = (
        paired_inference_results["mcnemar_exact_p_value"] < 0.05
    )
    paired_inference_results = paired_inference_results.sort_values(
        "observed_balanced_accuracy_gap", ascending=False
    ).reset_index(drop=True)
    display(
        paired_inference_results[[
            c for c in [
                "model_family",
                "numerical_model",
                "combined_model",
                "observed_balanced_accuracy_gap",
                "cluster_bootstrap_ba_ci95_low",
                "cluster_bootstrap_ba_ci95_high",
                "numerical_only_correct",
                "combined_only_correct",
                "mcnemar_exact_p_value",
                "mcnemar_holm_adjusted_p_value",
                "mcnemar_raw_reject_0_05",
                "mcnemar_holm_reject_0_05",
            ] if c in paired_inference_results.columns
        ]]
    )
    print("\nMcNemar multiple-testing summary:")
    print(
        "Raw p < 0.05:",
        int(paired_inference_results["mcnemar_raw_reject_0_05"].sum()),
        "of", len(paired_inference_results),
    )
    print(
        "Holm-adjusted p <= 0.05:",
        int(paired_inference_results["mcnemar_holm_reject_0_05"].sum()),
        "of", len(paired_inference_results),
    )
    paired_inference_results.to_csv(
        OUTPUT_DIR/"paired_company_cluster_inference_v8_1_holm.csv",
        index=False,
    )

correctness_cols=[c for c in [
    "observation_id",company_id_column,"quarter_end","target_clean"
] if c in preds.columns]
paired_correctness=preds[correctness_cols].copy()

for family,(num_name,comb_names) in comparison_specs.items():
    num_col=f"{num_name}_prediction"
    if num_col not in preds.columns:
        continue
    paired_correctness[f"{num_name}_correct"] = (
        preds[num_col].astype(int)==preds["target_clean"].astype(int)
    )
    for comb_name in comb_names:
        comb_col=f"{comb_name}_prediction"
        if comb_col in preds.columns:
            paired_correctness[f"{comb_name}_correct"] = (
                preds[comb_col].astype(int)==preds["target_clean"].astype(int)
            )

paired_correctness.to_csv(
    OUTPUT_DIR/"paired_row_correctness_v8.csv",index=False
)


## Paper-ready incremental-text summary


In [ ]:

if "bootstrap_results" in globals() and not bootstrap_results.empty:
    paper_incremental_text_table = bootstrap_results[
        [
            "model_family",
            "numerical_model",
            "combined_model",
            "n_test",
            "observed_balanced_accuracy_gap",
            "balanced_accuracy_ci95_low",
            "balanced_accuracy_ci95_high",
            "ba_ci_excludes_zero",
        ]
    ].copy()

    display(paper_incremental_text_table)

    paper_incremental_text_table.to_csv(
        OUTPUT_DIR / "paper_incremental_text_summary_v7.csv",
        index=False,
    )


## Formal multimodal ablation study

This is the main paper-level modality ablation. Compare numerical-only,
text-only, and fused representations. **Balanced accuracy is the primary
classification comparison**; Brier score is retained as a secondary measure
of probability quality/calibration.


In [ ]:
source_results = all_combined_results if "all_combined_results" in globals() else results
ablation_columns=[c for c in ["model","experiment_family","model_family","rows","accuracy","balanced_accuracy","macro_f1","acceleration_recall","deceleration_recall","roc_auc","brier_score","brier_skill_score","balanced_accuracy_delta_vs_same_row_numerical","selected_C","selected_params","threshold","selected_threshold","selected_pca_components"] if c in source_results.columns]
multimodal_ablation_results=source_results[ablation_columns].copy()
display(multimodal_ablation_results)


## v8 unified paper results package


In [ ]:
paper_workbook=OUTPUT_DIR/"paper_results_package_v8_1_holm.xlsx"
with pd.ExcelWriter(paper_workbook,engine="openpyxl") as writer:
    all_combined_results.to_excel(writer,sheet_name="All_Models",index=False)
    MASTER_COMPARISON_ROWS.to_excel(writer,sheet_name="Equal_Row_Manifest",index=False)
    preds.to_excel(writer,sheet_name="Paired_Test_Predictions",index=False)
    if "paired_inference_results" in globals():
        paired_inference_results.to_excel(writer,sheet_name="Paired_Inference",index=False)
    if "paired_correctness" in globals():
        paired_correctness.to_excel(writer,sheet_name="Row_Correctness",index=False)
    if "independent_validation_selection" in globals():
        independent_validation_selection.to_excel(writer,sheet_name="Validation_Selections",index=False)
    if "data_audit" in globals():
        data_audit.to_excel(writer,sheet_name="Data_Audit",index=False)
print("Saved:",paper_workbook)


## Robustness — unseen-company generalization

This secondary test reuses already-generated Sentence Transformer / FinBERT
features and tests them on a company that was never present in training.

In [ ]:
if not RUN_LOCO:
    print("LOCO skipped (set RUN_LOCO=True for the paper robustness run).")
else:
    company_col = next(
        (c for c in ["cik", "ticker", "company_name"] if c in text_model_data.columns),
        None,
    )
    if company_col is None:
        raise ValueError("No company identifier column found.")
    
    # Use only numeric model features. In particular, exclude finbert_text_hash
    # and use the actual Sentence Transformer column prefix text_embedding_.
    base_text_feature_columns = (
        [c for c in text_model_data.columns if c.startswith("text_embedding_")]
        + [
            c for c in [
                "finbert_positive_mean",
                "finbert_neutral_mean",
                "finbert_negative_mean",
                "finbert_negative_max",
                "finbert_negative_std",
                "finbert_positive_minus_negative",
            ]
            if c in text_model_data.columns
        ]
    )
    
    delta_feature_columns = [
        c for c in text_model_data.columns if c.endswith("_delta1")
    ]
    
    loco_feature_columns = list(
        dict.fromkeys(base_text_feature_columns + delta_feature_columns)
    )
    
    # Final safety check: LOCO must never receive non-numeric/hash/string columns.
    loco_feature_columns = [
        c for c in loco_feature_columns
        if pd.api.types.is_numeric_dtype(text_model_data[c])
    ]
    
    if not loco_feature_columns:
        print("Run Sentence Transformer / FinBERT feature generation first.")
    else:
        from sklearn.impute import SimpleImputer
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.linear_model import LogisticRegression
    
        print("LOCO numeric feature columns:", len(loco_feature_columns))
    
        loco_rows = []
        for company_value in sorted(text_model_data[company_col].dropna().unique()):
            held_out = text_model_data[
                (text_model_data[company_col] == company_value)
                & text_model_data["target_clean"].notna()
            ].copy()
            training_pool = text_model_data[
                (text_model_data[company_col] != company_value)
                & text_model_data["target_clean"].notna()
            ].copy()
    
            if len(held_out) < 2 or training_pool["target_clean"].nunique() < 2:
                continue
    
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(
                training_pool[loco_feature_columns],
                training_pool["target_clean"],
            )
            p = pipe.predict_proba(held_out[loco_feature_columns])[:, 1]
            y = held_out["target_clean"].astype(int)
            pred = (p >= 0.5).astype(int)
    
            row = evaluation_row(
                f"LOCO {company_value}",
                y,
                p,
                pred,
                base_p=float(training_pool["target_clean"].mean()),
            )
            row[company_col] = company_value
            row["n_test_rows"] = len(held_out)
            loco_rows.append(row)
    
        company_generalization_results = pd.DataFrame(loco_rows)
        if not company_generalization_results.empty:
            display(company_generalization_results)
            print("Companies evaluated:", len(company_generalization_results))
